# 05 — Sequential Behavior

- **Shaping**: trials-to-criterion per step, success rate, step acquisition curves
- **Chaining**: link completion rates, chain-break location, inter-link latencies

Run `00_data_pipeline` first.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT     = Path('..').resolve()
PROC_DIR = ROOT / 'data' / 'processed'
FIG_DIR  = ROOT / 'reports' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11,
                     'axes.spines.top': False, 'axes.spines.right': False})

def load(name):
    p = PROC_DIR / f'{name}.parquet'
    return pd.read_parquet(p) if p.exists() else pd.DataFrame()


## 1. Shaping — trials-to-criterion per step


In [ ]:
shp = load('shaping')

STEP_LABELS = {
    1: 'Step 1\n(platform)', 2: 'Step 2\n(small jump)',
    3: 'Step 3\n(gap clear)', 4: 'Step 4\n(moving platform)',
    5: 'Step 5\n(precision)'
}

if not shp.empty:
    # CRITERION events mark step transitions; their Note field contains
    # the step number (e.g., "step 3 -> 4")
    # STEP_UP events record which step became active
    # RESPONSE + SR_APPROX (success) and FAIL track trial outcomes

    shp = shp.sort_values(['participant_id', 'session_id', 'elapsed_s'])

    trial_rows = []
    for (pid, sid), grp in shp.groupby(['participant_id', 'session_id']):
        grp = grp.reset_index(drop=True)
        current_step = 1
        trial_num    = 0

        for _, row in grp.iterrows():
            if row['event_code'] == 'STEP_UP':
                current_step += 1
            elif row['event_code'] == 'RESPONSE':
                trial_num += 1
            elif row['event_code'] in ('SR_APPROX', 'FAIL'):
                trial_rows.append({
                    'participant_id': pid,
                    'session_id':     sid,
                    'step':           current_step,
                    'trial_num':      trial_num,
                    'elapsed_s':      row['elapsed_s'],
                    'outcome':        'success' if row['event_code'] == 'SR_APPROX' else 'fail',
                })

    if trial_rows:
        trial_df = pd.DataFrame(trial_rows)

        # Trials to criterion per step: number of trials before first 3-consecutive successes
        ttc_rows = []
        for (pid, step), grp in trial_df.groupby(['participant_id', 'step']):
            outcomes = grp['outcome'].tolist()
            # Count trials until 3 consecutive successes
            consec = 0
            for i, o in enumerate(outcomes):
                if o == 'success':
                    consec += 1
                    if consec >= 3:
                        ttc_rows.append({'participant_id': pid, 'step': step,
                                         'trials_to_criterion': i + 1,
                                         'success_rate': sum(1 for x in outcomes if x=='success')/len(outcomes)})
                        break
                else:
                    consec = 0
            else:
                # Criterion not met
                ttc_rows.append({'participant_id': pid, 'step': step,
                                  'trials_to_criterion': np.nan,
                                  'success_rate': sum(1 for x in outcomes if x=='success')/len(outcomes)})

        ttc_df = pd.DataFrame(ttc_rows)
        display(ttc_df.groupby('step').agg(
            mean_ttc=('trials_to_criterion', 'mean'),
            mean_success_rate=('success_rate', 'mean'),
            n=('trials_to_criterion', 'count')
        ).round(3))

        # Plot: trials-to-criterion by step
        fig, ax = plt.subplots(figsize=(8, 4))
        for pid, pgrp in ttc_df.groupby('participant_id'):
            pgrp = pgrp.sort_values('step')
            ax.plot(pgrp['step'], pgrp['trials_to_criterion'],
                    marker='o', label=pid)
        ax.set_xticks(list(STEP_LABELS.keys()))
        ax.set_xticklabels(list(STEP_LABELS.values()), fontsize=9)
        ax.set_ylabel('Trials to criterion')
        ax.set_title('Shaping — Trials to Criterion by Step')
        ax.legend()
        fig.savefig(FIG_DIR / 'shaping_trials_to_criterion.png', bbox_inches='tight')
        plt.show()


## 2. Chaining — link completion rates


In [ ]:
chn = load('chaining')

LINK_LABELS = {
    'LINK_1': 'L1: P-switch 1',
    'LINK_2': 'L2: ON/OFF switch',
    'LINK_3': 'L3: P-switch 2',
    'LINK_4': 'L4: Bridge/Flag'
}

if not chn.empty:
    chn = chn.sort_values(['participant_id', 'session_id', 'elapsed_s'])

    # Each run = sequence from start to CHAIN_COMPLETE or CHAIN_BREAK
    # Assign trial numbers per participant
    chn['trial_num'] = chn.groupby('participant_id').cumcount() + 1

    # Count link completions per trial
    link_codes = list(LINK_LABELS.keys())
    link_df = chn[chn['event_code'].isin(link_codes + ['CHAIN_COMPLETE', 'CHAIN_BREAK'])]

    # Trial-level summary
    trial_summaries = []
    current_trial = {'pid': None, 'links': [], 'start': None}

    for _, row in chn.iterrows():
        if row['event_code'] == 'LINK_1':
            current_trial = {'pid': row['participant_id'],
                             'links_completed': 0,
                             'break_at': None,
                             'start_t': row['elapsed_s'],
                             'link_times': []}
        if row['event_code'] in link_codes and current_trial['pid'] == row['participant_id']:
            current_trial['links_completed'] += 1
            current_trial['link_times'].append(row['elapsed_s'])
        elif row['event_code'] == 'CHAIN_BREAK' and current_trial['pid'] == row['participant_id']:
            current_trial['break_at'] = row['note']  # note contains "link X"
            trial_summaries.append(dict(current_trial))
            current_trial = {'pid': None}
        elif row['event_code'] == 'CHAIN_COMPLETE' and current_trial['pid'] == row['participant_id']:
            current_trial['break_at'] = None
            trial_summaries.append(dict(current_trial))
            current_trial = {'pid': None}

    if trial_summaries:
        trial_s_df = pd.DataFrame(trial_summaries)
        trial_s_df['completed'] = trial_s_df['break_at'].isna()

        print('=== Chain completion rate by participant ===')
        display(trial_s_df.groupby('pid')['completed']
                          .agg(['sum','count','mean'])
                          .rename(columns={'sum':'chains_completed','count':'trials','mean':'completion_rate'})
                          .round(3))

        # Break location distribution
        breaks = trial_s_df[trial_s_df['break_at'].notna()]
        if not breaks.empty:
            print('\n=== Chain break locations ===')
            display(breaks['break_at'].value_counts())

        # Link completion rates (proportion of trials reaching each link)
        fig, ax = plt.subplots(figsize=(7, 4))
        for i in range(1, 5):
            rate = (trial_s_df['links_completed'] >= i).mean()
            ax.bar(i, rate, color='#457b9d', alpha=0.85, edgecolor='white')
        ax.set_xticks(range(1, 5))
        ax.set_xticklabels([LINK_LABELS.get(f'LINK_{i}', f'Link {i}') for i in range(1, 5)],
                            rotation=15, fontsize=9)
        ax.set_ylabel('Proportion of trials reaching link')
        ax.set_ylim(0, 1.1)
        ax.set_title('Behavioral Chain — Link Completion Rates')
        fig.savefig(FIG_DIR / 'chaining_link_completion.png', bbox_inches='tight')
        plt.show()


## 3. Inter-link latencies


In [ ]:
if 'trial_s_df' in dir() and not trial_s_df.empty:
    # Extract link-to-link intervals from link_times
    latency_rows = []
    for _, row in trial_s_df.iterrows():
        times = row.get('link_times', [])
        if isinstance(times, list) and len(times) > 1:
            for i in range(len(times) - 1):
                latency_rows.append({
                    'participant_id': row['pid'],
                    'interval': f'L{i+1}→L{i+2}',
                    'latency_s': times[i+1] - times[i]
                })

    if latency_rows:
        lat_df = pd.DataFrame(latency_rows)
        display(lat_df.groupby('interval')['latency_s']
                       .agg(['mean','std','count'])
                       .rename(columns={'mean':'mean_s','std':'sd_s','count':'n'})
                       .round(3))

        fig, ax = plt.subplots(figsize=(7, 4))
        for interval, grp in lat_df.groupby('interval'):
            ax.bar(interval, grp['latency_s'].mean(),
                   yerr=grp['latency_s'].sem(), capsize=4,
                   alpha=0.8, color='#2a9d8f')
        ax.set_ylabel('Latency (s, mean ± SEM)')
        ax.set_title('Behavioral Chain — Inter-Link Latencies')
        fig.savefig(FIG_DIR / 'chaining_interlink_latencies.png', bbox_inches='tight')
        plt.show()
